# Executing quantum programs

Read finite samples and exact states, apply a calibration-derived local noise model, and choose an installed simulator.

## Separate intent, compilation and execution

The quantum function below allocates a three-bit input and prepares it uniformly,
then computes a fresh y=x*x+1 while preserving x. All eight input values are
present in one coherent program. The compiler infers the output width and builds
reversible arithmetic within the declared budgets. You do not supply multiplier
gates, carry wires or inverse cleanup.

`fq.run` executes the program. The first named readout is an exact simulator
probability distribution: eight `(x,x*x+1)` pairs, each of probability 1/8.
The next sections deliberately use finite shots, so their frequencies fluctuate.
[workflow.py](workflow.py) contains the sampling, noise and backend plumbing.
Its finite-count adapter currently maps named register locations to low-level
counts and decodes them; this is not a new public named-sampling synth API.

Try replacing `x**2+1` with `x+1` and inspect the resources and exact outcomes.
The existing independent square reference checks below intentionally apply to
the original expression; update a reference independently when changing the task.


Synthesize the coherent square-plus-one function and inspect its resource estimate.

In [ ]:
import flagquantum as fq
from flagquantum import synth
from flagquantum.synth import Output, QNum, allocate, hadamard_transform, qfunc

@qfunc
def square_plus_one(x: Output[QNum], y: Output[QNum]) -> None:
    allocate(3, x)
    hadamard_transform(x)
    y |= x**2 + 1

from examples.synthesis.tutorials.basics.quantum_programming.execution_tutorial.overview import main, workflow

square = synth.synthesize(
    square_plus_one, max_qubits=10, max_workspace=1, memory_limit_bytes=1024**2,
)
exact = dict(square.probabilities(fq.run(square), registers=("x", "y")))
print("Exact named outputs:", exact)
print(f"Square circuit resources: {square.report['resources']}")

Draw 2,048 actual samples from the compiled arithmetic circuit.

In [ ]:
default = workflow.square_counts(square, workflow.DEFAULT_SHOTS)

Read the named input and output values.

In [ ]:
main.print_square(default)

Repeat with 10,000 shots.

In [ ]:
more = workflow.square_counts(square, workflow.LARGE_SHOTS)

Read the larger sample and check all eight coherent amplitudes and the work bit.

In [ ]:
main.print_square(more)
main.print_square_error(square)

Prepare the three-qubit GHZ program with native Python qfunc.

In [ ]:
ghz = workflow.build_ghz()

Run a local density-matrix model with Pittsburgh snapshot data on physical chain 0–1–2. Gate noise and measurement assignment errors are verified separately.

In [ ]:
main.print_noisy(ghz)

Inspect the ideal complex statevector.

In [ ]:
main.print_ideal(ghz)

Discover installed local execution backends.

In [ ]:
print(f"Installed local backends: {', '.join(workflow.backend_names())}")

Run the requested statevector and MPS modes independently.

In [ ]:
default_state = workflow.ideal_state(ghz)
mps = workflow.mps_counts(ghz)

Check both nonzero amplitudes in the explicit default-mode result.

In [ ]:
assert complex(default_state[0]) == complex(default_state[7])
assert default_state.abs().count_nonzero().item() == 2

Read the actual backend-native MPS counts.

In [ ]:
main.print_mps(mps)